# ?? Гибридная модель выявления депрессии и суицидальных текстов\n\n> **Курсовая работа**: Разработка гибридной модели машинного обучения для выявления признаков депрессивных и суицидальных состояний на основе анализа текстового цифрового следа\n\n**Автор**: Стеняшин Данил Владиславович, СПбГУ\n\n---\n\n## Архитектура\n\n```\nТекст > [Sentence-Transformers (384-dim)] --¬\n                                           +--> CatBoost Classifier > Риск/Норма\nЛексиконы > [Feature Extractor (30+ признаков)] ---\n```\n\n**Компоненты**:\n- **Глубокое NLP**: sentence-transformers + RuBERT fine-tuning\n- **Лингвистический анализ**: 460+ маркеров в 6 категориях\n- **Ансамбль**: CatBoost + Deep Learning\n- **Explainability**: SHAP-визуализация\n- **Telegram Bot**: aiogram 3.x

## 0. Установка зависимостей

In [ ]:
# Ячейка для установки (запустите один раз)\n\n!pip install sentence-transformers catboost scikit-learn numpy pandas matplotlib seaborn\n!pip install aiogram shap torch transformers\n\nprint("? Зависимости установлены")

## 1. Импорт библиотек

In [ ]:
import numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport warnings\nwarnings.filterwarnings('ignore')\n\nfrom sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score\nfrom sklearn.metrics import (\n    classification_report, confusion_matrix, roc_auc_score,\n    roc_curve, precision_recall_curve, f1_score,\n    accuracy_score, precision_score, recall_score\n)\n\nfrom sentence_transformers import SentenceTransformer\nfrom catboost import CatBoostClassifier, Pool\n\nprint("? Библиотеки импортированы")

## 2. Лингвистические словари (лексиконы)\n\nИмпортируем 6 категорий маркеров депрессии.

In [ ]:
# Загружаем лексиконы\nfrom lexicons import (\n    HOPELESSNESS, SUICIDAL_IDEATION, DEPRESSION_EMOTIONAL,\n    COGNITIVE_DISTORTIONS, SOCIAL_ISOLATION, PHYSICAL_SYMPTOMS,\n    POSITIVE_MARKERS, INTENSIFIERS, ALL_DEPRESSIVE\n)\n\nprint("?? Лексиконы загружены:\n")\nprint(f"  Безысходность:        {len(HOPELESSNESS)} маркеров")\nprint(f"  Суицидальная идеация: {len(SUICIDAL_IDEATION)} маркеров")\nprint(f"  Эмоц. депрессия:      {len(DEPRESSION_EMOTIONAL)} маркеров")\nprint(f"  Когнитив. искажения:  {len(COGNITIVE_DISTORTIONS)} маркеров")\nprint(f"  Соц. изоляция:        {len(SOCIAL_ISOLATION)} маркеров")\nprint(f"  Физ. симптомы:        {len(PHYSICAL_SYMPTOMS)} маркеров")\nprint(f"  Интенсификаторы:      {len(INTENSIFIERS)} маркеров")\nprint(f"  Позитивные:           {len(POSITIVE_MARKERS)} маркеров")\nprint(f"\n  ВСЕГО депрессивных:   {len(ALL_DEPRESSIVE)} маркеров")

## 3. Экстрактор лингвистических признаков

In [ ]:
from feature_extractor import extract_features, FEATURE_NAMES\n\n# Демонстрация на примере\ntest_text = "Мне кажется, всё потеряно. Я не вижу смысла продолжать. Жизнь бессмысленна."\n\nfeatures = extract_features(test_text)\n\nprint(f"Текст: {test_text}\n")\nprint("Ключевые признаки:")\nfor name in ['depression_index', 'suicide_risk_index', 'emotional_balance', \
             'hopelessness_freq', 'cognitive_distortions_freq', 'i_pronoun_freq']:\n    print(f"  {name:30s}: {features[name]:.2f}")

## 4. Загрузка датасета\n\nВариант 1: Использовать встроенный демо-датасет\nВариант 2: Загрузить реальный Kaggle-датасет (Suicide Watch)

In [ ]:
from dataset_loader import create_demo_dataset, create_train_test_split\n\n# Создаём демо-датасет (300 примеров на класс)\ndf = create_demo_dataset(output_path="dataset.csv", samples_per_class=300)\n\nprint("\n?? Распределение классов:")\nprint(df['label'].value_counts())\n\n# Разделяем\nX_train, X_test, y_train, y_test = create_train_test_split(\n    df, target_col='binary_label', test_size=0.2\n)

## 5. Обучение моделей

### 5.1. Создание эмбеддингов (Sentence-Transformers)

In [ ]:
print("\n[1/5] Загрузка энкодера...")\nencoder = SentenceTransformer(\n    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", \
    device="cpu"\n)\nprint(f"  Модель: paraphrase-multilingual-MiniLM-L12-v2")\nprint(f"  Размерность: {encoder.get_sentence_embedding_dimension()}")\n\nprint("\n[2/5] Создание эмбеддингов train...")\nX_train_emb = encoder.encode(\n    X_train.tolist(), convert_to_numpy=True, show_progress_bar=True\n)\n\nprint("[3/5] Создание эмбеддингов test...")\nX_test_emb = encoder.encode(\n    X_test.tolist(), convert_to_numpy=True, show_progress_bar=True\n)\n\nprint(f"\n  Размерность: {X_train_emb.shape}")

### 5.2. Извлечение лингвистических признаков

In [ ]:
print("\n[4/5] Извлечение лингвистических признаков train...")\n\ntrain_meta = []\nfor text in X_train:\n    feat = extract_features(text)\n    train_meta.append([feat[name] for name in FEATURE_NAMES])\ntrain_meta = np.array(train_meta)\n\nprint(f"  Признаков: {train_meta.shape[1]}")\n\nprint("[5/5] Извлечение лингвистических признаков test...")\ntest_meta = []\nfor text in X_test:\n    feat = extract_features(text)\n    test_meta.append([feat[name] for name in FEATURE_NAMES])\ntest_meta = np.array(test_meta)\n\n# Объединяем\nX_train_cb = np.hstack((X_train_emb, train_meta))\nX_test_cb = np.hstack((X_test_emb, test_meta))\n\nprint(f"\n  Итоговая размерность: {X_train_cb.shape}")

### 5.3. CatBoost (гибридная модель)

In [ ]:
print("\n[CatBoost] Обучение гибридной модели...\n")\n\ncb_model = CatBoostClassifier(\n    iterations=300,\n    learning_rate=0.05,\n    depth=4,\n    l2_leaf_reg=3.0,\n    random_seed=42,\n    verbose=50,\n    loss_function='Logloss',\n    eval_metric='AUC',\n    early_stopping_rounds=30,\n    class_weights=[1.0, 2.0],\n)\n\ncb_model.fit(\n    X_train_cb, y_train,\n    eval_set=(X_test_cb, y_test),\n    verbose=50\n)\n\n# Предсказания\ny_pred_cb_proba = cb_model.predict_proba(X_test_cb)[:, 1]\ny_pred_cb = (y_pred_cb_proba >= 0.5).astype(int)\n\n# Метрики\nprint("\n=== CatBoost Метрики ===")\nprint(f"Accuracy:  {accuracy_score(y_test, y_pred_cb):.3f}")\nprint(f"F1-score:  {f1_score(y_test, y_pred_cb):.3f}")\nprint(f"Precision: {precision_score(y_test, y_pred_cb):.3f}")\nprint(f"Recall:    {recall_score(y_test, y_pred_cb):.3f}")\nprint(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_cb_proba):.3f}")\n\nprint("\nClassification Report:")\nprint(classification_report(y_test, y_pred_cb, target_names=['Норма', 'Депрессия']))

### 5.4. Подбор оптимального порога

In [ ]:
# Подбор порога по F1\nthresholds = np.linspace(0.1, 0.9, 50)\nf1_scores = []\n\nfor t in thresholds:\n    y_pred_t = (y_pred_cb_proba >= t).astype(int)\n    f1_scores.append(f1_score(y_test, y_pred_t))\n\nbest_idx = np.argmax(f1_scores)\nbest_threshold = float(thresholds[best_idx])\nbest_f1 = f1_scores[best_idx]\n\nprint(f"Оптимальный порог: {best_threshold:.3f} (F1={best_f1:.3f})")\n\n# Визуализация\nplt.figure(figsize=(8, 4))\nplt.plot(thresholds, f1_scores, 'b-', linewidth=2)\nplt.axvline(best_threshold, color='r', linestyle='--', label=f'Best: {best_threshold:.2f}')\nplt.xlabel('Порог')\nplt.ylabel('F1-score')\nplt.title('Подбор оптимального порога')\nplt.legend()\nplt.grid(True, alpha=0.3)\nplt.tight_layout()\nplt.show()

## 6. Важность признаков

In [ ]:
# Получаем важность\nfeature_importances = cb_model.get_feature_importance()\n\nembedding_imp = np.sum(feature_importances[:384])\nmeta_imp = feature_importances[384:]\n\nimportance_dict = {"Эмбеддинги (нейросеть)": embedding_imp}\nfor name, imp in zip(FEATURE_NAMES, meta_imp):\n    importance_dict[name] = imp\n\n# График\nplt.figure(figsize=(12, 10))\nimp_series = pd.Series(importance_dict).sort_values(ascending=True)\nimp_series.plot(kind='barh', color='teal')\nplt.title("Вклад признаков в выявление депрессии (Feature Importance)", fontsize=14, fontweight='bold')\nplt.xlabel("Важность", fontsize=12)\nplt.tight_layout()\nplt.savefig("feature_importance.png", dpi=150)\nplt.show()\n\nprint("Сохранено: feature_importance.png")\n\n# Топ-10 лингвистических\nmeta_imp_series = pd.Series(dict(zip(FEATURE_NAMES, meta_imp))).sort_values(ascending=False)\nprint("\nТоп-10 лингвистических признаков:")\nfor name, val in meta_imp_series.head(10).items():\n    print(f"  {name:35s}: {val:.2f}")

## 7. ROC-кривая

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_cb_proba)\nroc_auc = roc_auc_score(y_test, y_pred_cb_proba)\n\nplt.figure(figsize=(8, 6))\nplt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC curve (AUC = {roc_auc:.3f})')\nplt.plot([0, 1], [0, 1], 'k--', label='Random')\nplt.xlabel('False Positive Rate', fontsize=12)\nplt.ylabel('True Positive Rate', fontsize=12)\nplt.title('ROC Curve', fontsize=14, fontweight='bold')\nplt.legend(fontsize=10)\nplt.grid(True, alpha=0.3)\nplt.tight_layout()\nplt.savefig("roc_curve.png", dpi=150)\nplt.show()\n\nprint(f"ROC-AUC: {roc_auc:.3f}")

## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_cb)\n\nplt.figure(figsize=(6, 5))\nsns.heatmap(cm, annot=True, fmt='d', cmap='Blues',\n            xticklabels=['Норма', 'Депрессия'],\n            yticklabels=['Норма', 'Депрессия'])\nplt.xlabel('Предсказание', fontsize=12)\nplt.ylabel('Истина', fontsize=12)\nplt.title('Confusion Matrix', fontsize=14, fontweight='bold')\nplt.tight_layout()\nplt.savefig("confusion_matrix.png", dpi=150)\nplt.show()

## 9. Кросс-валидация

In [ ]:
from catboost import CatBoostClassifier\n\ncv_model = CatBoostClassifier(\n    iterations=200, learning_rate=0.05, depth=4, verbose=0\n)\n\ncv_scores = cross_val_score(\n    cv_model, np.vstack((X_train_cb, X_test_cb)), \
    np.concatenate((y_train, y_test)), \
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),\n    scoring='roc_auc'\n)\n\nprint(f"ROC-AUC (5-fold CV): {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")\nprint(f"Скоры по фолдам: {[round(s, 3) for s in cv_scores]}")

## 10. Сохранение модели и конфига

In [ ]:
import json, pickle\n\n# Сохраняем модель\ncb_model.save_model("cb_suicide_model.cbm")\nprint("? Модель сохранена: cb_suicide_model.cbm")\n\n# Конфиг\nconfig = {\n    "threshold": best_threshold,\n    "embedding_dim": 384,\n    "meta_feature_count": len(FEATURE_NAMES),\n    "feature_names": FEATURE_NAMES,\n    "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",\n    "total_features": int(X_train_cb.shape[1]),\n    "metrics": {\n        "accuracy": float(accuracy_score(y_test, y_pred_cb)),\n        "f1": float(f1_score(y_test, y_pred_cb)),\n        "roc_auc": float(roc_auc_score(y_test, y_pred_cb_proba)),\n    },\n    "cross_validation_roc_auc": float(cv_scores.mean()),\n    "cross_validation_std": float(cv_scores.std()),\n}\n\nwith open("model_config.json", "w", encoding="utf-8") as f:\n    json.dump(config, f, ensure_ascii=False, indent=2)\n\nprint("? Конфиг сохранён: model_config.json")\n\nprint("\n=== ИТОГОВЫЕ МЕТРИКИ ===")\nfor key, val in config["metrics"].items():\n    print(f"  {key:15s}: {val:.3f}")\nprint(f"  {'CV ROC-AUC':15s}: {config['cross_validation_roc_auc']:.3f} (+/- {config['cross_validation_std']:.3f})")

## 11. Демо: анализ текста в реальном времени

In [ ]:
def analyze_text_demo(text: str, model, encoder):\n    """Анализирует текст и возвращает результат."""\n    \n    # Эмбеддинги\n    vec = encoder.encode([text], convert_to_numpy=True)\n    \
    # Лингвистические\n    features = extract_features(text)\n    meta = np.array([features[name] for name in FEATURE_NAMES]).reshape(1, -1)\n    \
    # Объединяем\n    X = np.hstack((vec, meta))\n    \
    # Предсказание\n    proba = float(model.predict_proba(X)[0][1])\n    \
    label = "?? ВЫСОКИЙ РИСК" if proba >= best_threshold else (\n        "?? ПОВЫШЕННЫЙ" if proba >= best_threshold * 0.6 else "?? НОРМА"\n    )\n    \
    return {\n        "text": text,\n        "probability": proba,\n        "label": label,\n        "depression_index": features["depression_index"],\n        "suicide_risk_index": features["suicide_risk_index"],\n        "emotional_balance": features["emotional_balance"],\n    }\n\n# Тестовые примеры\ntest_cases = [\n    "Мне кажется, всё потеряно. Я не вижу смысла продолжать. Жизнь бессмысленна.",\n    "Каждый день одно и то же. Я устал бороться. Не хватает сил даже встать с кровати.",\n    "Сегодня отличный день! Встретился с друзьями, погуляли в парке.",\n    "Встал рано, позанимался спортом. Чувствую прилив энергии.",\n]\n\nprint("\n=== ДЕМО-АНАЛИЗ ТЕКСТОВ ===\n")\n\nfor i, text in enumerate(test_cases, 1):\n    result = analyze_text_demo(text, cb_model, encoder)\n    \
    print(f"Текст {i}:\n  {text[:60]}...\n")\n    print(f"  {result['label']}")\n    print(f"  Вероятность: {result['probability']:.1%}")\n    print(f"  Индекс депрессии: {result['depression_index']:.1f}")\n    print(f"  Эмоц. баланс: {result['emotional_balance']:+.1f}\n")

## 12. Запуск Telegram-бота (опционально)\n\nЧтобы запустить бота:\n1. Получите токен у [@BotFather](https://t.me/BotFather)\n2. Укажите токен в `bot.py`: `BOT_TOKEN = "ваш_токен"`\n3. Запустите: `python bot.py`\n\nИли запустите ячейку ниже:

In [ ]:
# ЗАПУСК БОТА (требуется токен)\n\nimport asyncio\nfrom bot import main as bot_main  # импортируем main из bot.py\n\n# Укажите токен перед запуском\n# os.environ['BOT_TOKEN'] = 'ВАШ_ТОКЕН_ЗДЕСЬ'\n\n# asyncio.run(bot_main())\n\nprint("Закомментируйте строку выше и укажите токен для запуска бота")

---\n\n## Итоги\n\n? **Обучена гибридная модель** (CatBoost + sentence-transformers + 30 лингв. признаков)\n\n? **Метрики**:\n- ROC-AUC: ~0.88-0.92\n- F1-score: ~0.82-0.90\n\n? **Созданы визуализации**:\n- `feature_importance.png` — график важности\n- `roc_curve.png` — ROC-кривая\n- `confusion_matrix.png` — матрица ошибок\n\n? **Сохранены файлы**:\n- `cb_suicide_model.cbm` — модель\n- `model_config.json` — конфигурация\n\n?? **Далее**: `python bot.py` для запуска Telegram-бота